[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeteoSwiss/opendata-nwp-demos/blob/main/05_interpolate_vertically.ipynb)


# Vertical Interpolation of ICON-CH2-EPS Temperature Forecasts

This notebook demonstrates the full workflow for accessing ICON-CH2-EPS temperature forecasts and performing **vertical interpolation from model levels to either pressure levels or a target altitude**. The data is provided by MeteoSwiss as part of Switzerland’s [Open Government Data (OGD) initiative](https://www.meteoswiss.admin.ch/services-and-publications/service/open-data.html).

The core functionality is provided by the [earthkit ecosystem](https://earthkit.readthedocs.io/en/latest/), developed by ECMWF to simplify loading, processing, and visualising numerical weather prediction data. The MeteoSwiss [earthkit-data-meteoswiss-opendata](https://meteoswiss.github.io/earthkit-data-meteoswiss-opendata/) configures a custom [earthkit source](https://earthkit-data.readthedocs.io/en/latest/concepts/plugins/sources_plugin.html) for accessing MeteoSwiss NWP Open Data. Once loaded through earthkit, the data can be processed using temporal and spatial operators and visualised with earthkit’s meteorological plotting tools, such as horizontal or vertical interpolation.

The ICON-CH1/2-EPS data is typically provided on terrain-following vertical model levels, which do not correspond directly to atmospheric pressure levels. For consistent comparison and analysis, it is often necessary to interpolate these forecasts to pressure levels. In some applications, interpolation to a specific altitude is also useful.

The [earthkit-meteo](https://earthkit-meteo.readthedocs.io/en/latest/) library provides two functions for this purpose: 
- [interpolate_to_pressure_levels()](https://earthkit-meteo.readthedocs.io/en/latest/autoapi/earthkit/meteo/vertical/xarray/interpolate_to_pressure_levels.html): interpolates a field from model levels to pressure coordinates. 
- [interpolate_monotonic()](https://earthkit-meteo.readthedocs.io/en/latest/autoapi/earthkit/meteo/vertical/xarray/interpolate_monotonic.html): can be used to interpolate a field from model levels to a fixed altitude.

---

## 🔍 **What You’ll Do in This Notebook**

 🛰️  **Retrieve**  
    Fetch deterministic ICON-CH2-EPS forecast data (temperature (`T`) and pressure (`P`)) using [earthkit-data](https://earthkit-data.readthedocs.io/en/latest/) with the [earthkit-data-meteoswiss-opendata](https://meteoswiss.github.io/earthkit-data-meteoswiss-opendata/) source plugin.

 📈  **Vertical Interpolation using [earthkit-meteo vertical module](https://earthkit-meteo.readthedocs.io/en/latest/autoapi/earthkit/meteo/vertical/index.html).**  
 -  **Interpolate to pressure levels**:
    Interpolate ICON-CH2-EPS forecast data to pressure levels.

 -  **Interpolate to target altitude**:
    Interpolate ICON-CH2-EPS forecast data to a specific target altitude.

---

## Retrieving Forecasts
In this first part, we retrieve deterministic ICON-CH2-EPS temperature and pressure forecast data. To access this data, we use [earthkit-data](https://earthkit-data.readthedocs.io/en/latest/) to load deterministic ICON-CH2-EPS precipitation forecasts from MeteoSwiss Open Data. The earthkit-data-meteoswiss-opendata plugin provides the custom earthkit source that translates the request into a query to the MeteoSwiss [STAC (SpatioTemporal Asset Catalog) API](https://data.geo.admin.ch/api/stac/static/spec/v1/api.html#tag/STAC/operation/postSearchSTAC) and resolves the matching forecast assets.

### 📁  Browsing the STAC Catalog (Optional)

The STAC Catalog provides structured access to Switzerland’s open geospatial data.
If you'd like to explore the ICON-CH1/2-EPS forecast datasets interactively before writing code, you can browse them directly in the STAC catalog:

&nbsp;&nbsp;&nbsp;&nbsp;🔗  [Browse the ICON-CH1-EPS collection](https://data.geo.admin.ch/browser/#/collections/ch.meteoschweiz.ogd-forecasting-icon-ch1?.language=en)

&nbsp;&nbsp;&nbsp;&nbsp;🔗  [Browse the ICON-CH2-EPS collection](https://data.geo.admin.ch/browser/#/collections/ch.meteoschweiz.ogd-forecasting-icon-ch2?.language=en)


Below is a screenshot of the ICON-CH2-EPS collection as seen in the STAC browser interface.


![browser-ch2.png](./images/browser-ch2.png)

⚙️ Notebook & Eccodes Definition Setup

To run the notebook in Google Colab the required dependencies need to be installed. It is skipped in a local Jupyter environment, where dependencies are assumed to be installed already.

Moreover, MeteoSwiss uses custom ecCodes definitions for ICON data. To correctly interpret MeteoSwiss-specific `shortName` values, you must configure the definition path. 

In [ ]:
# 📦 Notebook setup
import sys, os, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/MeteoSwiss/opendata-nwp-demos.git
    %cd opendata-nwp-demos

    !pip install poetry && poetry config virtualenvs.in-project true && poetry install --no-ansi

    venv = pathlib.Path(".venv")
    site = venv / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    sys.path.insert(0, str(site))

    # set eccodes definition path
    os.environ["ECCODES_DEFINITION_PATH"] = str((venv / "share/eccodes-cosmo-resources/definitions").resolve())

else:
    import eccodes_cosmo_resources

    # set eccodes definition path
    os.environ["ECCODES_DEFINITION_PATH"] = str(
        eccodes_cosmo_resources.get_definitions_path()
    )

os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

### Creating Requests
n this part, we define which forecast data to load by specifying the model collection, variable, forecast run, and lead times. Earthkit passes these parameters to the MeteoSwiss source plugin, which builds the corresponding STAC query and retrieves the matching assets. In this example, we create two requests: one to retrieve the temperature field (`T`), which we aim to interpolate, and another to obtain the pressure field (`P`) on model levels, required for the interpolation.

>⏰ **Forecast Availability**: Forecast data will typically be available a couple of hours after the reference time — due to the model runtime and subsequent upload time. The data remains accessible for 24 hours after upload.

In [4]:
param_list = ["T", "P"]
req_list = []

for param in param_list:
    req = {
        "collection": "ogd-forecasting-icon-ch2",
        "variable": param,
        "ref_time": "latest",
        "perturbed": False,
        "lead_time": "P0DT0H"
    }
    req_list.append((param,req))

Each argument in the request serves the following purpose:

| Argument             | Description |
|----------------------|-------------|
| `collection`         | Forecast collection to use (e.g., `ogd-forecasting-icon-ch2` for ICON-CH2-EPS). |
| `variable`           | Meteorological variable of interest (`T` = temperature and `P` = pressure). |
| `ref_time` | Initialization time of the forecast in **UTC**, provided as either:<br>- The string `"latest"` to select the newest forecast run (`ref_time`) that contains all requested lead times. All assets returned by a single request therefore share the same `ref_time`. Be cautious: separate requests (for example, for different variables) resolve `"latest"` independently and may return different `ref_time` values while MeteoSwiss is uploading a new forecast run. <br>- [datetime.datetime](https://docs.python.org/3/library/datetime.html#datetime-objects) object (e.g.,<br> &nbsp; `datetime.datetime(2025, 5, 22, 9, 0, 0, tzinfo=datetime.timezone.utc)`) <br>- [ISO 8601](https://en.wikipedia.org/wiki/ISO_8601#Combined_date_and_time_representations) date string (e.g., `"2025-05-22T09:00:00Z"`)|
| `perturbed`          | If `True`, retrieves ensemble forecast members; if `False`, returns the deterministic forecast. |
| `lead_time`            | Forecast lead time, provided as either:<br>– [datetime.timedelta](https://docs.python.org/3/library/datetime.html#timedelta-objects) object (e.g., `datetime.timedelta(hours=0)`) <br>– [ISO 8601](https://en.wikipedia.org/wiki/ISO_8601#Durations) duration string (e.g., `"P0DT0H"`)|

### Retrieving Data
We now pass the desired request to the MeteoSwiss earthkit source. The plugin builds the STAC query, and earthkit retrieves the matching forecast data.
Each response is returned as an **[xarray.Dataset](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.html)**, which is efficient for handling multi-dimensional data.

> 💡 **Tip**: Use temporary caching with earthkit-data to skip repeated downloads — it's auto-cleaned after the session.
> *For more details, see the [earthkit-data caching docs](https://earthkit-data.readthedocs.io/en/latest/examples/cache.html)*.

> 💡 **Hint**: If you get an error message containing `HTTPError: 403 Client Error: Forbidden for url`, you may be trying to retrieve data older than 24h hours! Please adjust your requests.

In [5]:
import earthkit.data as ekd
ekd.config.set("cache-policy", "temporary")

data_dict = {}
for param, request in req_list:

    forecast = ekd.from_source(
        "meteoswiss-opendata",
        **request,
        )

    data_dict[param] = forecast.to_xarray(
        time_dims=["forecast_reference_time", "step"],
        squeeze=False
        )

The resulting `xarray.Dataset` has the following dimensions:

- `member` (ensemble members): 0 (for unperturbed data)
- `forecast_reference_time`: single reference time (the latest available run)
- `step`: single lead time (e.g. +0 hours)
- `level`: vertical level ((e.g. 80 model levels))
- `level_type`: type of vertical level (e.g. model level)
- `values`: 283,876 spatial grid points

The spatial grid is represented by the one-dimensional `values` dimension. `latitude` and `longitude` are coordinates indexed by `values`, so each grid
point corresponds to one latitude–longitude pair. They are therefore coordinates of the data rather than separate dataset dimensions.

In [6]:
data_dict["T"]

<xarray.Dataset> Size: 186MB
Dimensions:                  (member: 1, forecast_reference_time: 1, step: 1,
                              level: 80, level_type: 1, values: 283876)
Coordinates:
  * member                   (member) <U1 4B '0'
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2026...
  * step                     (step) timedelta64[ns] 8B 00:00:00
  * level                    (level) int64 640B 1 2 3 4 5 6 ... 76 77 78 79 80
  * level_type               (level_type) <U7 28B 'general'
    latitude                 (values) float64 2MB ...
    longitude                (values) float64 2MB ...
Dimensions without coordinates: values
Data variables:
    T                        (member, forecast_reference_time, step, level, level_type, values) float64 182MB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

💡 To prepare for interpolation, let's inspect the initial vertical coordinate, the **model levels**:

In [7]:
data_dict["T"].coords["level"]

<xarray.DataArray 'level' (level: 80)> Size: 640B
array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
       37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
       55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72,
       73, 74, 75, 76, 77, 78, 79, 80])
Coordinates:
  * level    (level) int64 640B 1 2 3 4 5 6 7 8 9 ... 72 73 74 75 76 77 78 79 80
Attributes:
    long_name:  general
    units:      1
    positive:   down

### Model levels

As shown in the output above, the parameter `T` includes temperature values on **80 model levels**. These model levels represent how the atmosphere is discretised from the Earth's surface up to the top of the model domain. In the numerical weather model ICON the 80 model levels correspond to the so-called **full levels** and are numbered top down. They are following the terrain and gradually change into levels of constant height as the distance from the surface increases (see the picture below). For more information about model levels, refer to the [model grid documentation](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model#vertical-grid).

<div style="text-align: center;">
  <img src="./images/VerticalLayers.png" width="50%">
</div>

## Vertical Interpolation
We perform two types of vertical interpolation: interpolating forecast data to pressure levels and to a target altitude.

### Interpolation from Model to Pressure Levels

Forecast data is typically stored on model (or hybrid) levels, on which the pressure varies with the atmospheric state. For comparison, plotting, or diagnostics, it's often meaningful to interpolate this data to **pressure levels**. The [interpolate_to_pressure_levels()](https://earthkit-meteo.readthedocs.io/en/latest/autoapi/earthkit/meteo/vertical/xarray/interpolate_to_pressure_levels.html) function from the **earthkit-meteo** toolbox performs this transformation.

---
#### How `interpolate_to_pressure_levels()` Works

To compute the interpolated value $f(p_t)$ at a target pressure $p_t$, the function `interpolate_to_pressure_levels()`:

1. **Identifies two adjacent model levels** $k$ and $k-1$ such that:
   - $p_1 = \text{pressure}[k-1]$ is just **above** $p_t$
   - $p_2 = \text{pressure}[k]$ is just **below** $p_t$

2. **Retrieves the field values (here temperature)** at those levels:
   - $f_1 = \text{field}[k-1]$
   - $f_2 = \text{field}[k]$

3. **Computes the interpolation ratio** $r$ depending on the selected method:

    - **Linear Interpolation (`linear_in_p`)**
    $$
    r = \frac{p_t - p_1}{p_2 - p_1}, \quad f(p_t) = (1 - r) f_1 + r f_2
    $$

    - **Linear Interpolation w.r.t Log-Pressure (`linear_in_lnp`)**
    $$
    r = \frac{\ln(p_t) - \ln(p_1)}{\ln(p_2) - \ln(p_1)}, \quad f(p_t) = (1 - r) f_1 + r f_2
    $$

    - **Nearest Model Surface/Level (`nearest_sfc`)**
    $$
    f(p_t) =
    \begin{cases}
    f_1 & \text{if } |p_t - p_1| < |p_t - p_2| \\
    f_2 & \text{otherwise}
    \end{cases}
    $$

This logic is applied point-wise across the full 3D field.

#### Comparison of Interpolation Methods

Each interpolation method produces slightly different results. For example, if you have temperature values at model levels *k* and *k-1* and want to estimate the value at 600 hPa, the result will depend on the method used. The following example illustrates how each of the three methods generates different outputs.

<div style="text-align: center;">
  <img src="./images/interpolation.png" width="50%">
</div>

#### Implementation
The choice of interpolation method depends on the relationship between the variable's vertical gradient and the pressure profile. In this example, we use logarithmic interpolation (`"log"`) for temperature data, which is appropriate for variables that vary approximately linearly with the logarithm of pressure.

The `interpolate_to_pressure_levels()` function requires the following arguments:
- `data`: parameter to be interpolated
- `p`: pressure levels (in Pa)
- `target_p`: target pressure level values
- `target_p_units`: the unit of the target pressure levels (`"Pa"` or `"hPa"`)
- `interpolation`: interpolation algorithm (`"linear"`, `"log"` or `"nearest"`)
- `vertical_dim`: the name of the vertical dimension

In [8]:
import earthkit.meteo.vertical as vertical

target_levels = [500, 550, 600, 650, 700, 750, 800, 850]

T_interpolated = vertical.interpolate_to_pressure_levels(
    data=data_dict["T"]["T"],
    p=data_dict["P"]["P"],
    target_p=target_levels,
    target_p_units="hPa",
    interpolation="log",
    vertical_dim="level"
    )

#### Pressure Level Inspection
After interpolation, the parameter now has updated vertical levels:
- `level`: 50000 - 85000 Pa (vertical levels)

In [9]:
coords = T_interpolated.level
print(f"\033[1mlevel \033[0m(vertical levels): {coords.values}")

level (vertical levels): [50000. 55000. 60000. 65000. 70000. 75000. 80000. 85000.]


However, to accurately describe the interpolated `xarray.DataArray`, we need to update its metadata.

In [10]:
T_interpolated.attrs["level_type"] = "pressure level"
T_interpolated.attrs

{'standard_name': 'air_temperature',
 'long_name': 'Temperature',
 'units': 'kelvin',
 'level_type': 'pressure level',
 '_earthkit': '{"message": {"__bytes_b64__": "R1JJQv//AAIAAAAAAAAAxwAAABUBANcA/w8BAQfqBx8MAAABBQAAAB4CAP0AB+oHHw04OQAAAAAAAAAAAQABAAD//wAAACMDAAAEVOQAAABlBgAAAgG7vVoJhVSZJDx6SqTIdikgAAAAPQQABgABAAAEAI4AAAAAAAAAAJYAAAAAAZYAAAAAAsAAFUKiAABAgAAATQG4hxocgBwDFvKA8uYswAAAABUFAARU5AAAQ4iAAIAKAAAAAAAAAAYG/wAAAAUHNzc3Nw=="}, "bitsPerValue": 16, "grid_spec": {"grid": "ICON-CH2_C", "uid": "bbbd5a09855499243c7a4aa4c8762920"}}'}

### Interpolation from Model Levels to Target Altitude

Instead of interpolating to pressure levels, it is also possible to interpolate to a specific target altitude. This transformation can be performed using **earthkit-meteo**'s [`interpolate_monotonic()`](https://earthkit-meteo.readthedocs.io/en/latest/autoapi/earthkit/meteo/vertical/xarray/interpolate_monotonic.html) function.

---
#### How `interpolate_monotonic()` Works

The function `interpolate_monotonic()` is the general form of `interpolate_to_pressure_levels()`: passing the height of the model levels (or any field that varies monotonically with height) as the vertical coordinate instead of the pressure field lets us interpolate our data to a target altitude rather than a target pressure.

#### Implementation

In this example, we demonstrate how to interpolate temperature data on model levels to a specific target altitude.

The `interpolate_monotonic()` function requires the following arguments:
- `data`: parameter to be interpolated
- `coords`: height field on model levels
- `target_coords`: target coordinates
- `interpolation`: interpolation algorithm (`"linear"`, `"log"` or `"nearest"`)
- `vertical_dim`: the name of the vertical dimension

To interpolate to a specific altitude the argument `data` and `coords` require the data on the same height levels. Here the height above sea level on **full model level (HFL)**. To derive the HFL data, we first retrieve the vertical grid parameters that provide the height above sea level at **half levels (HHL)**. Next, we perform a destaggering operation on these half levels to compute the corresponding heights at the full levels.

In [11]:
# retrieve HHL
HHL = ekd.from_source(
    "meteoswiss-opendata-constants",
    collection="ogd-forecasting-icon-ch2",
    asset="vertical",
).to_xarray()

# temporary workaround until the earthkit implementation is available
hhl_rename = HHL.rename({"level": "z"})
hhl_rename = hhl_rename.assign_coords({"eps": 0, "lead_time": 0, "ref_time": 0})
hhl_prep = hhl_rename.HHL
hhl_prep = hhl_prep.assign_attrs({"vcoord_type": "model_level", "origin_z": -0.5, 'message_b64': 'R1JJQv//AAIAAAAAAAAAxwAAABUBANcA/w8BAQfqBx8GAAABBQAAAB4CAP0AB+oHHwc4KgAAAAAAAAAAAQABAAD//wAAACMDAAAEVOQAAABlBgAAAgG7vVoJhVSZJDx6SqTIdikgAAAAPQQABgABAwYEAI4AAAAAAAAAAJYAAAAAAWUAAAAAAMAAFUKiAABAgAAATQG4hxocgBwDFvKA8uYswAAAABUFAARU5AAARqvgAIAKAAAAAAAAAAYG/wAAAAUHNzc3Nw=='})

# use meteodata-lab until earthkit provides destaggering
from meteodatalab.operators.destagger import destagger

HFL = destagger(hhl_prep.squeeze(drop=True), "z")

# temporaray workaround until the earthkit implementation is available
HFL = HFL.rename({"z": "level"})

Once retrieved the HFL data, continue with the interpolation.

In [12]:
T_height = vertical.interpolate_monotonic(
    data=data_dict["T"]["T"],
    coords=HFL,
    target_coords=[1000, 2000],
    interpolation="linear",
    vertical_dim="level"
)

#### Target Altitude Inspection
After interpolation, the parameter now has updated vertical levels:
- `level`: [1000 2000] masl (2 vertical levels)

In [13]:
coords = T_height.level
print(f"\033[1mlevel \033[0m(vertical levels): {coords.values}")

level (vertical levels): [1000 2000]
